# 1.1. Install and import dependencies

In [ ]:
!pip install poetry
!python3 -m pip install --upgrade pip setuptools wheel

  Using cached pip-24.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached setuptools-72.1.0-py3-none-any.whl.metadata (6.6 kB)
Using cached pip-24.2-py3-none-any.whl (1.8 MB)
Using cached setuptools-72.1.0-py3-none-any.whl (2.3 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 71.0.4
    Uninstalling setuptools-71.0.4:
      Successfully uninstalled setuptools-71.0.4
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
pyproject_content = """[tool.poetry]
name = "lab_1_example"
description = "Training pipeline for deep learning model"
authors = ["Name Surname <name.surname@gmail.com>"]
version = "0.01"

[tool.poetry.dependencies]
python = "~3.10"
torch = { version = "2.1.0+cu118", source = "pytorch" }
torchvision = { version = "0.16.0+cu118", source = "pytorch" }
tqdm = "4.64.1"
matplotlib = "3.6.3"
numpy = "1.22.4"
pyyaml = "6.0"
scipy = "1.13.0rc1"
pandas = ">2.0"

[tool.poetry.dev-dependencies]
mypy = "0.991"
ruff = "0.0.254"
black = "23.1.0"
isort = "5.12.0"

[build-system]
requires = ["poetry-core>=1.0.0"]
build-backend = "poetry.core.masonry.api"

[[tool.poetry.source]]
name = "pytorch"
priority = "supplemental"
url = "https://download.pytorch.org/whl/cu118" """

# Write the content to pyproject.toml
with open("pyproject.toml", "w") as file:
    file.write(pyproject_content)

# Step 2: Install dependencies using pip
!poetry export --without-hashes --output requirements.txt
!pip install -r requirements.txt

In order to avoid a breaking change and make your automation forward-compatible, please install poetry-plugin-export explicitly. See https://python-poetry.org/docs/plugins/#using-plugins for details on how to install a plugin.
To disable this warning run 'poetry config warnings.export false'.
The lock file does not exist. Locking.
Updating dependencies
Resolving dependencies... (7.3s)

Writing lock file
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
Ignoring colorama: markers 'python_version >= "3.10" and python_version < "3.11" and platform_system == "Windows"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 90.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.

In [ ]:
import zipfile
import tarfile
import os
import logging
import requests

from pathlib import Path
from typing import Optional, Tuple, Dict, Any, Union, List

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset
from torchvision.io import read_image
from torchvision import transforms

import scipy.io as sio

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# hack to fix logging in colab https://stackoverflow.com/a/74121821/11709937
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(level=logging.INFO)

# 1.2 Data download (same as lab 1)

In [ ]:
def download_and_extract(url: str, save_dir: str, filename: Optional[str] = None) -> str:
    """
    Downloads a file from the given URL and extracts it if it is an archive.

    Args:
    - url (str): The URL of the file to download.
    - save_dir (str): Directory where the file will be saved and extracted.
    - filename (Optional[str]): Optional filename to save the file with. If None, uses the filename from the URL.

    Returns:
    - str: The full path to the directory containing the extracted files or the downloaded file.
    """
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)

    if filename is None:
        filename = url.split("/")[-1]

    file_path = save_path / filename
    result_path = str(file_path)

    # Check if file already exists
    if file_path.exists():
        logging.info(f"File '{filename}' already exists in '{save_dir}'. Skipping download.")
    else:
        logging.info(f"Downloading file '{filename}' from '{url}'...")
        try:
            response = requests.get(url)
            response.raise_for_status()
            with open(file_path, 'wb') as f:
                f.write(response.content)
            logging.info(f"Download successful. File saved to: '{file_path}'")

            # Extract the file if it's an archive
            if filename.endswith(".zip"):
                with zipfile.ZipFile(file_path, 'r') as zip_ref:
                    zip_ref.extractall(save_path)
                file_path.unlink()
                result_path = str(save_path)
            elif (
                  filename.endswith(".tar.gz")
                  or filename.endswith(".tgz")
                  or filename.endswith(".gz")
                ):
                with tarfile.open(file_path, 'r:gz') as tar_ref:
                    tar_ref.extractall(save_path)
                file_path.unlink()
                result_path = str(save_path)
            elif filename.endswith(".tar"):
                with tarfile.open(file_path, 'r') as tar_ref:
                    tar_ref.extractall(save_path)
                file_path.unlink()
                result_path = str(save_path)
            else:
                logging.info(f"File '{filename}' is not an archive. No extraction needed.")

        except requests.RequestException as e:
            logging.error(f"Error downloading file from {url}: {str(e)}")
            raise
        except (zipfile.BadZipFile, tarfile.ReadError) as e:
            logging.error(f"Error extracting file '{filename}': {str(e)}")
            raise

    return result_path

# 1.3 Data ingestion (same as lab 1)

In [ ]:
def train_test_split(
    data: pd.DataFrame,
    test_size: Union[float, int] = 0.25,
    random_state: Union[int, None] = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split arrays or matrices into random train and test subsets.

    Parameters:
    X :  pd.DataFrame
         The input data to split.
    test_size : float, int, or None, optional (default=0.25)
        If float, should be between 0.0 and 1.0 and represent the proportion of the dataset to include in the test split.
        If int, represents the absolute number of test samples.
        If None, the value is set to the complement of the train size.
    random_state : int or None, optional (default=None)
        If int, random_state is the seed used by the random number generator;
        If None, the random number generator is the RandomState instance used by np.random.

    Returns:
    Tuple containing:
        - data_train: pd.DataFrame
        - data_test: pd.DataFrame
    """
    if random_state is not None:
        np.random.seed(random_state)

    n_samples = len(data)
    if isinstance(test_size, float):
        test_size = int(n_samples * test_size)

    indices = np.random.permutation(n_samples)
    test_indices = indices[:test_size]
    train_indices = indices[test_size:]

    data_train = data.iloc[train_indices]
    data_test = data.iloc[test_indices]

    return data_train, data_test


def load_labels(labels_path: str) -> pd.DataFrame:
    """
    Load labels from a MAT file.

    Args:
    - labels_path (str): Path to the labels MAT file.

    Returns:
    - pd.DataFrame: DataFrame containing the labels.
    """
    labels_mat = sio.loadmat('/content/data/labels/imagelabels.mat')
    labels_df = pd.DataFrame({"label": labels_mat["labels"][0]})

    return labels_df

def find_add_images_to_labels(images_dir: str, labels: pd.DataFrame, image_ext: str="jpg") -> pd.DataFrame:
    image_paths = sorted(
        [str(image_path.absolute()) for image_path in Path(images_dir).rglob(f"*.{image_ext}")]
    )
    if len(image_paths) != len(labels):
        logging.error(
            f"Found {len(image_paths)} image_paths but "
            f"{len(labels)} labels were provided, cannot continue."
        )
        raise ValueError

    labels_w_image_paths = labels.copy(deep=True)
    labels_w_image_paths["image_path"] = image_paths
    return labels_w_image_paths

# 1.3.1. Data ingestion with batches (new)

In [ ]:
def assign_batches(labels_df: pd.DataFrame, config: Dict[str, Any]) -> pd.DataFrame:
    """
    Assigns batch labels to a dataframe.

    Args:
    - labels_df (pd.Dataframe): Dataframe containing dataset information in 'label' and 'image_path' columns.
    - config (Dict[str, Any]): Configuration dictionary containing parameters for data splitting and paths.

    Returns:
    - pd.DataFrame: dataframe with assigned batch labels in 'batch_name' column.
    """
    labels_df_ = labels_df.copy(deep=True)
    labels_df_["batch_name"] = "not_set"

    n_batches = config["n_batches"]
    batch_size = len(labels_df_) // n_batches

    batch_size_current = 0
    for batch_number in range(n_batches):
        if batch_number == (n_batches -1):
            # select all the remaining data for the last batch
            labels_df_.iloc[
                    batch_size_current:,
                    labels_df_.columns.get_loc('batch_name')
            ] = str(batch_number)
        else:
            labels_df_.iloc[
                batch_size_current: batch_size_current + batch_size,
                labels_df_.columns.get_loc('batch_name')
            ] = str(batch_number)

        batch_size_current += batch_size

    return labels_df_


def select_batches(labels_df: pd.DataFrame, config: Dict[str, Any]) -> pd.DataFrame:
    """
    Selects data from labels_df based on 'batch_names' from config.

    Args:
    - labels_df (pd.Dataframe): .
    - config (Dict[str, Any]): Configuration dictionary containing parameters for data splitting and paths.

    Returns:
    - pd.DataFrame:
    """
    batch_names: List[str] = config["batch_names_select"]
    labels_df_ = labels_df.copy(deep=True)

    labels_df_ = labels_df_[labels_df_["batch_name"].isin(batch_names)]

    return labels_df_


def process_data(images_dir: str, labels_path: str, config: Dict[str, Any]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Process the data by loading labels, validating data, splitting into train-validation-test sets,
    and optionally saving the splits.

    Args:
    - images_dir (str): Directory containing the images.
    - labels_path (str): Path to the labels MAT file.
    - config (Dict[str, Any]): Configuration dictionary containing parameters for data splitting and paths.

    Returns:
    - Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]: Training, validation, and testing DataFrames.
    """
    # Load labels
    labels_df = load_labels(labels_path)

    # Validate data (if needed, adjust based on your specific validation logic)
    valid_labels_df = find_add_images_to_labels(images_dir, labels_df)

    # Split data into train and test
    train_df, test_df = train_test_split(valid_labels_df, test_size=config.get('test_size', 0.2), random_state=config.get('random_state', 42))

    train_df = assign_batches(train_df, config)
    train_df = select_batches(train_df, config)

    # Further split train_df into train and validation
    train_df, val_df = train_test_split(train_df, test_size=config.get('val_size', 0.2), random_state=config.get('random_state', 42))

    logging.info(f"Prepared 3 data splits: train, size: {len(train_df)}, val: {len(val_df)}, test: {len(val_df)}")

    return train_df, val_df, test_df

# Example run

In [ ]:
def main():
    # Configuration
    config = {
        'test_size': 0.2,
        'val_size': 0.2,
        'random_state': 42,
        "lr": 0.001,
        "n_batches": 6,
        "batch_names_select": ["0", "1", "2", "3", "4", "5"]
    }
    # Step 1: Download data
    images_url = 'https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz'
    labels_url = 'https://www.robots.ox.ac.uk/~vgg/data/flowers/102/imagelabels.mat'
    images_dir = download_and_extract(images_url, '/content/data/images')
    labels_path = download_and_extract(labels_url, '/content/data/labels')

    # Step 2: Process data
    train_df, val_df, test_df = process_data(images_dir, labels_path, config)

    logging.info(
        f"TRAIN dataset size: {len(train_df)}, "
        f"VAL dataset size: {len(val_df)}, "
        f"Test dataset size: {len(test_df)}"
    )

SyntaxError: unterminated string literal (detected at line 9) (<ipython-input-1-0944feb3c2c3>, line 9)

In [ ]:
main()

INFO:root:Downloading file '102flowers.tgz' from 'https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz'...
INFO:root:Download successful. File saved to: '/content/data/images/102flowers.tgz'
INFO:root:File 'imagelabels.mat' already exists in '/content/data/labels'. Skipping download.
INFO:root:Prepared 3 data splits: train, size: 1748, val: 436, test: 436
INFO:root:TRAIN dataset size: 1748, VAL dataset size: 436, Test dataset size: 1637
